# Assignment 4 — Spark Fundamentals (Olist)

Consolidated notebook built from the two supplied Spark notebooks and aligned to the DataKern Assignment 4 PDF.

**Important:** run this notebook from inside the Olist repository. It locates `data/raw/` automatically and writes generated files to the repository's `output/` directory. The Parquet write uses a normal local filesystem path (not a `file:///...` URI), then reads the output back to verify it.

Run cells from top to bottom. Spark UI screenshots must be captured from your own run; do not submit fabricated metrics or screenshots.

## Initial answers (before coding)
1. **Two filters then count:** I expect the source to be read **once**. Spark joins both filters into one plan. Reading again can happen if I run another action without cache.
2. **DAG:** It shows the steps of the work and which step depends on which, so Spark can order and split the work.
3. **Stage vs task:** A stage is a group of steps that can run without a shuffle. A task is one stage running on one partition.
4. **filter vs groupBy:** filter looks at each row alone, so no data moves. groupBy needs all rows with the same key together, so data must move (shuffle).

## Part 1 - Environment, SparkSession and data

In [38]:
import os, sys, subprocess, time
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python :", sys.version.split()[0])
try:
    r = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print("Java   :", (r.stderr or r.stdout).splitlines()[0])
except FileNotFoundError:
    print("Java not found. Install a Java version compatible with your PySpark version, then restart the kernel.")

import pyspark
print("PySpark:", pyspark.__version__)

# Find the repository root even when Jupyter starts from the project root,
# the notebooks/ directory, or a nested working directory.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
ROOT = next((p for p in candidates if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "Could not find data/raw. Open this notebook from your Olist repository "
        "and check that the required CSV files are in data/raw/."
    )

RAW = (ROOT / "data" / "raw").as_posix()
OUT = ROOT / "output"
OUT.mkdir(parents=True, exist_ok=True)

required_files = [
    "olist_orders_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_products_dataset.csv",
    "olist_product_category_name_translation.csv",
]
missing = [name for name in required_files if not (ROOT / "data" / "raw" / name).is_file()]
if missing:
    raise FileNotFoundError("Missing required Olist CSV file(s): " + ", ".join(missing))

print("Project root:", ROOT)
print("Raw data directory:", RAW)
print("Output directory (generated locally; do not commit):", OUT)

Python : 3.12.1
Java   : openjdk version "17.0.20.1" 2026-08-18
PySpark: 4.2.0
Project root: C:\Users\PAVANI\OneDrive\Desktop\OneDrive\Desktop\olist-pandas-assignment
Raw data directory: C:/Users/PAVANI/OneDrive/Desktop/OneDrive/Desktop/olist-pandas-assignment/data/raw
Output directory (generated locally; do not commit): C:\Users\PAVANI\OneDrive\Desktop\OneDrive\Desktop\olist-pandas-assignment\output


In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import (
    col, year, month, to_timestamp, date_format, coalesce, lit, when, broadcast,
    countDistinct, sum as spark_sum, round as spark_round # type: ignore
)
import pandas as pd

spark = (
    SparkSession.builder
    .appName("DataKern-Olist-Spark")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")   # small data, so 8 instead of default 200
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark UI:", sc.uiWebUrl)

Spark UI: http://Kiranmayi1814:4040


**Spark UI before any data action** - take screenshot now (Jobs page should be empty).

- `local[*]` = run Spark on this machine. `*` = use all CPU cores as worker threads.
- Jobs page is empty because no action has run yet.

### Helper: logs jobs, stages and tasks for every step (fills the Spark UI log)

In [8]:
log = []

def plan_text(df):
    try:
        return df._jdf.queryExecution().executedPlan().toString()
    except Exception:
        return ""

def run_step(step, action, fn, df=None, cache="No", note=""):
    group = "step_%d" % (len(log) + 1)
    sc.setJobGroup(group, step)
    t0 = time.time()
    result = fn()
    secs = round(time.time() - t0, 2)
    time.sleep(0.5)
    tr = sc.statusTracker()
    jobs = list(tr.getJobIdsForGroup(group))
    stages = set()
    for j in jobs:
        info = tr.getJobInfo(j)
        if info:
            stages.update(info.stageIds)
    tasks = 0
    for s in stages:
        si = tr.getStageInfo(s)
        if si:
            tasks += si.numTasks
    shuffle = "Yes" if (len(stages) > 1 or (df is not None and "Exchange" in plan_text(df))) else "No"
    log.append({"Step": step, "Action": action, "Jobs": len(jobs), "Stages": len(stages),
                "Tasks": tasks, "Shuffle/Exchange?": shuffle, "Cache?": cache,
                "Seconds": secs, "Observation": note})
    return result

### 1.1 Schema inference vs explicit schema

In [9]:
orders_inferred = run_step(
    "Infer schema (orders)", "read.csv(inferSchema=True)",
    lambda: spark.read.csv(RAW + "/olist_orders_dataset.csv", header=True, inferSchema=True),
    note="Reading with inferSchema already created a job (it scanned the file).")
orders_inferred.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [10]:
def str_schema(names):
    return StructType([StructField(n, StringType(), True) for n in names])

orders_schema = str_schema([
    "order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"])

customers_schema = str_schema([
    "customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"])

order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True),
])

products_schema = StructType(
    [StructField("product_id", StringType(), True), StructField("product_category_name", StringType(), True)] +
    [StructField(n, DoubleType(), True) for n in [
        "product_name_lenght", "product_description_lenght", "product_photos_qty",
        "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]])

translation_schema = str_schema(["product_category_name", "product_category_name_english"])

def read_csv(fname, schema):
    return spark.read.csv(RAW + "/" + fname, header=True, schema=schema)

orders       = read_csv("olist_orders_dataset.csv", orders_schema)
customers    = read_csv("olist_customers_dataset.csv", customers_schema)
order_items  = read_csv("olist_order_items_dataset.csv", order_items_schema)
products     = read_csv("olist_products_dataset.csv", products_schema)
translation  = read_csv("olist_product_category_name_translation.csv", translation_schema)

for name, d in [("orders", orders), ("customers", customers), ("order_items", order_items),
                ("products", products), ("translation", translation)]:
    print("==", name)
    d.printSchema()

== orders
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

== customers
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

== order_items
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullab

- Inference **does read the file** to guess types. On big files this is slow because it scans a lot of data.
- Explicit schema = no scan, no wrong guesses, same types every run.
- `printSchema()` only shows the schema. It does not scan rows.

### 1.2 First action: count

In [11]:
n_orders = run_step("Count orders", "count()", lambda: orders.count(), df=orders,
                    note="Single scan of the CSV. Task count follows input file splits.")
print("Orders rows:", n_orders)

Orders rows: 99441


## Part 2 - Transformations and plans

### 2.1 select()
**Prediction:** `select()` alone creates **no job**.

In [12]:
orders_slim = orders.select("order_id", "customer_id", "order_status", "order_purchase_timestamp")
orders_slim.printSchema()
run_step("Select count", "count()", lambda: orders_slim.count(), df=orders_slim,
         note="Select is lazy. Job only started at count.")

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)



99441

`select()` keeps the columns you name. `drop()` removes the columns you name.

### 2.2 filter()
**Prediction:** about 96,000 delivered orders.

In [13]:
delivered_orders = orders_slim.filter(orders_slim.order_status == "delivered")
n_del = run_step("Delivered filter", "count()", lambda: delivered_orders.count(), df=delivered_orders,
                 note="Filter is narrow. Each task filters its own partition.")
print("Delivered orders:", n_del)

Delivered orders: 96478


- filter is narrow: each row is checked alone, no data moves.
- Task count shows how many partitions were read. It does **not** show how many rows match or how fast it is.

### 2.3 withColumn()
**Prediction:** 3 new columns. Chaining does **not** make separate jobs.

In [14]:
orders_with_month = (
    delivered_orders
    .withColumn("purchase_ts", to_timestamp(col("order_purchase_timestamp")))
    .withColumn("purchase_year", year(col("purchase_ts")))
    .withColumn("purchase_month", month(col("purchase_ts")))
)
print("New columns:", [c for c in orders_with_month.columns if c not in delivered_orders.columns])
orders_with_month.explain()

New columns: ['purchase_ts', 'purchase_year', 'purchase_month']
== Physical Plan ==
*(1) Project [order_id#81, customer_id#82, order_status#83, order_purchase_timestamp#84, purchase_ts#142, year(cast(purchase_ts#142 as date)) AS purchase_year#143, month(cast(purchase_ts#142 as date)) AS purchase_month#144]
+- *(1) Project [order_id#81, customer_id#82, order_status#83, order_purchase_timestamp#84, cast(order_purchase_timestamp#84 as timestamp) AS purchase_ts#142]
   +- *(1) Filter (isnotnull(order_status#83) AND (order_status#83 = delivered))
      +- FileScan csv [order_id#81,customer_id#82,order_status#83,order_purchase_timestamp#84] Batched: false, DataFilters: [isnotnull(order_status#83), (order_status#83 = delivered)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/PAVANI/OneDrive/Desktop/OneDrive/Desktop/olist-pandas-a..., PartitionFilters: [], PushedFilters: [IsNotNull(order_status), EqualTo(order_status,delivered)], ReadSchema: struct<order_id:string,customer_i

The plan shows one combined Project and Filter on one scan. All steps run in one pass.

### 2.4 join()
**Prediction:** row count **increases**. One order has many items, so joining order (1 row) to items (many rows) gives one row per item.

In [15]:
orders_with_items = orders_with_month.join(order_items, on="order_id", how="inner")
n_join = run_step("Join orders/items", "count()", lambda: orders_with_items.count(), df=orders_with_items,
                  note="Join needs data with the same order_id together, so Exchange appears unless broadcast.")
print("Joined rows:", n_join, "| delivered orders:", n_del)
orders_with_items.explain()

Joined rows: 110197 | delivered orders: 96478
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [order_id#81, customer_id#82, order_status#83, order_purchase_timestamp#84, purchase_ts#142, purchase_year#143, purchase_month#144, order_item_id#95, product_id#96, seller_id#97, shipping_limit_date#98, price#99, freight_value#100]
   +- SortMergeJoin [order_id#81], [order_id#94], Inner
      :- Sort [order_id#81 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(order_id#81, 8), ENSURE_REQUIREMENTS, [plan_id=322]
      :     +- Project [order_id#81, customer_id#82, order_status#83, order_purchase_timestamp#84, purchase_ts#142, year(cast(purchase_ts#142 as date)) AS purchase_year#143, month(cast(purchase_ts#142 as date)) AS purchase_month#144]
      :        +- Project [order_id#81, customer_id#82, order_status#83, order_purchase_timestamp#84, cast(order_purchase_timestamp#84 as timestamp) AS purchase_ts#142]
      :           +- Filter ((isnotnull(order_status

- **Sort Merge Join:** both sides are shuffled and sorted by key, then merged. Good for two big tables.
- **Broadcast Hash Join:** the small table is sent to every worker, so the big table does not move. Good when one side is small.
- Spark picks by table size (`autoBroadcastJoinThreshold`) and join type.

In [16]:
# Broadcast experiment: small lookup table = translation
items_prod = order_items.join(products.select("product_id", "product_category_name"), "product_id")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # turn off auto broadcast
print("---- WITHOUT broadcast ----")
items_prod.join(translation, "product_category_name", "left").explain()
print("---- WITH broadcast() ----")
items_prod.join(broadcast(translation), "product_category_name", "left").explain()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")  # back to default (10 MB)

---- WITHOUT broadcast ----
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_category_name#102, product_id#96, order_id#94, order_item_id#95, seller_id#97, shipping_limit_date#98, price#99, freight_value#100, product_category_name_english#111]
   +- SortMergeJoin [product_category_name#102], [product_category_name#110], LeftOuter
      :- Sort [product_category_name#102 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(product_category_name#102, 8), ENSURE_REQUIREMENTS, [plan_id=381]
      :     +- Project [product_id#96, order_id#94, order_item_id#95, seller_id#97, shipping_limit_date#98, price#99, freight_value#100, product_category_name#102]
      :        +- SortMergeJoin [product_id#96], [product_id#101], Inner
      :           :- Sort [product_id#96 ASC NULLS FIRST], false, 0
      :           :  +- Exchange hashpartitioning(product_id#96, 8), ENSURE_REQUIREMENTS, [plan_id=373]
      :           :     +- Filter isnotnull(product_id#96)
 

Broadcasting a **large** table is bad: it is copied to every worker and can run out of memory.

### 2.5 groupBy() and agg()

In [17]:
monthly_summary = (
    orders_with_items
    .groupBy("purchase_year", "purchase_month")
    .agg(countDistinct("order_id").alias("distinct_orders"),
         spark_round(spark_sum("price"), 2).alias("total_item_sales"))
    .orderBy("purchase_year", "purchase_month")
)
run_step("Monthly aggregation", "show()", lambda: monthly_summary.show(24), df=monthly_summary,
         note="groupBy and orderBy cause Exchange. Several stages.")
monthly_summary.explain()

+-------------+--------------+---------------+----------------+
|purchase_year|purchase_month|distinct_orders|total_item_sales|
+-------------+--------------+---------------+----------------+
|         2016|             9|              1|          134.97|
|         2016|            10|            265|        40325.11|
|         2016|            12|              1|            10.9|
|         2017|             1|            750|       111798.36|
|         2017|             2|           1653|        234223.4|
|         2017|             3|           2546|       359198.85|
|         2017|             4|           2303|       340669.68|
|         2017|             5|           3546|       489338.25|
|         2017|             6|           3135|       421923.37|
|         2017|             7|           3872|       481604.52|
|         2017|             8|           4193|        554699.7|
|         2017|             9|           4150|       607399.67|
|         2017|            10|          

- Look for `Exchange hashpartitioning` in the plan = shuffle.
- Rows with the same key must meet in one partition, or we cannot add them together.
- The redistributed data is the needed columns (keys and values).
- Too much shuffle = disk writes, network copy and waiting, so it is slow.

### 2.6 Narrow vs wide

| Operation | Dependency | Shuffle? |
|---|---|---|
| select / filter / withColumn | Narrow | No |
| groupBy / repartition | Wide | Yes |
| join | Depends on strategy | Often |

A shuffle splits a job into stages: stage 1 runs before the shuffle and writes its output. Stage 2 starts after, reading that output.

### 2.7 dropDuplicates()

In [18]:
def dup_check():
    total = order_items.count()
    distinct = order_items.dropDuplicates(["order_id", "order_item_id"]).count()
    return total, distinct

total_rows, distinct_rows = run_step("Deduplication", "count()", dup_check,
                                     note="dropDuplicates is wide (shuffle).")
print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Duplicate keys:", total_rows - distinct_rows)

order_items_clean = order_items.dropDuplicates(["order_id", "order_item_id"])

Total rows: 112650
Distinct rows: 112650
Duplicate keys: 0


In [19]:
# Controlled test (small fake data, source files not touched)
test_df = spark.createDataFrame(
    [("o1", 1, 10.0), ("o1", 1, 10.0), ("o1", 1, 99.0), ("o2", 1, 5.0)],
    ["order_id", "order_item_id", "price"])
print("No subset (whole row compared):")
test_df.dropDuplicates().show()
print("Subset (order_id, order_item_id):")
test_df.dropDuplicates(["order_id", "order_item_id"]).show()

No subset (whole row compared):
+--------+-------------+-----+
|order_id|order_item_id|price|
+--------+-------------+-----+
|      o1|            1| 10.0|
|      o1|            1| 99.0|
|      o2|            1|  5.0|
+--------+-------------+-----+

Subset (order_id, order_item_id):
+--------+-------------+-----+
|order_id|order_item_id|price|
+--------+-------------+-----+
|      o1|            1| 10.0|
|      o2|            1|  5.0|
+--------+-------------+-----+



- No subset: all columns are compared. Subset: only the listed columns.
- Same key but different values: Spark keeps **any one** row. We cannot control which.
- That is not safe for business data (e.g. it may keep the wrong price). Define the grain and a rule first.

## Part 3 - Actions, cache, partitions, Parquet

### 3.1 Common actions (on small `monthly_summary`)

In [20]:
r1 = run_step("Action count()", "count()", lambda: monthly_summary.count(), df=monthly_summary)
r2 = run_step("Action show(5)", "show(5)", lambda: monthly_summary.show(5), df=monthly_summary)
r3 = run_step("Action first()", "first()", lambda: monthly_summary.first(), df=monthly_summary)
r4 = run_step("Action take(3)", "take(3)", lambda: monthly_summary.take(3), df=monthly_summary)
r5 = run_step("Action collect() small", "collect()", lambda: monthly_summary.collect(), df=monthly_summary)
print(r1, r3, len(r5))
orders_with_items.show(5)   # small sample only. Never collect() this one.

+-------------+--------------+---------------+----------------+
|purchase_year|purchase_month|distinct_orders|total_item_sales|
+-------------+--------------+---------------+----------------+
|         2016|             9|              1|          134.97|
|         2016|            10|            265|        40325.11|
|         2016|            12|              1|            10.9|
|         2017|             1|            750|       111798.36|
|         2017|             2|           1653|        234223.4|
+-------------+--------------+---------------+----------------+
only showing top 5 rows
23 Row(purchase_year=2016, purchase_month=9, distinct_orders=1, total_item_sales=134.97) 23
+--------------------+--------------------+------------+------------------------+-------------------+-------------+--------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp

`collect()` on a big DataFrame brings **all rows to the driver** (one machine). It can run out of memory and crash. Use `show()` or `take()` for samples.

### 3.2 Cache the Finance base

In [21]:
finance_base = (
    orders.join(order_items, on="order_id", how="inner")
    .filter(col("order_status") == "delivered")
)
finance_base.cache()
n_fb = run_step("Cache finance_base", "count()", lambda: finance_base.count(), df=finance_base, cache="Yes",
                note="First count fills the cache. Later actions read from memory.")
print("Rows cached:", n_fb, "| is_cached:", finance_base.is_cached, "|", finance_base.storageLevel)

Rows cached: 110197 | is_cached: True | Disk Memory Deserialized 1x Replicated


**Take the Storage tab screenshot now** (cached DataFrame must be listed).

- If the cache does not fit in memory, Spark keeps some partitions and recomputes the rest (or spills to disk for MEMORY_AND_DISK).
- Do not reuse `finance_base` for Operations because it has only delivered orders.

### 3.3 repartition() and coalesce()

In [22]:
print("Original:", orders_with_items.rdd.getNumPartitions())
orders_repartitioned = orders_with_items.repartition(8)
print("After repartition:", orders_repartitioned.rdd.getNumPartitions())
orders_coalesced = orders_repartitioned.coalesce(2)
print("After coalesce:", orders_coalesced.rdd.getNumPartitions())

run_step("Repartition", "count()", lambda: orders_repartitioned.count(), df=orders_repartitioned,
         note="repartition(8) adds a shuffle.")
run_step("Coalesce", "count()", lambda: orders_coalesced.count(), df=orders_coalesced,
         note="coalesce merges partitions, so fewer tasks in last stage.")

Original: 8
After repartition: 8
After coalesce: 2


110197

- More partitions = more parallel tasks. Too few = cores sit idle. Too many = lots of tiny tasks and overhead.
- `repartition()` shuffles all data to make even partitions.
- `coalesce()` just merges nearby partitions, so less data moves.
- Too aggressive coalesce = few big tasks, less parallelism, uneven work.

### 3.4 Write Parquet

In [37]:
# Parquet write: use a normal local path. This avoids file-URI parsing issues on some Windows setups.
import platform
import pandas as pd

parquet_dir = OUT / "monthly_sales_summary"
out_path = str(parquet_dir.resolve())

def write_parquet():
    try:
        monthly_summary.write.mode("overwrite").parquet(out_path)
    except Exception as exc:
        message = str(exc)
        if platform.system() == "Windows" and any(
            marker.lower() in message.lower()
            for marker in ("HADOOP_HOME", "winutils", "checkHadoopHome")
        ):
            # Workaround for Windows installations without winutils.exe.
            # The summary is small, so collect only this aggregated DataFrame.
            try:
                import pyarrow  # noqa: F401
            except ImportError as pyarrow_error:
                raise RuntimeError(
                    "Spark Parquet writing failed because winutils.exe/HADOOP_HOME "
                    "is missing. Install pyarrow for the small-data fallback:\n"
                    "    pip install pyarrow"
                ) from pyarrow_error

            parquet_dir.mkdir(parents=True, exist_ok=True)
            fallback_file = parquet_dir / "part-00000.parquet"

            monthly_summary.toPandas().to_parquet(
                fallback_file,
                engine="pyarrow",
                index=False
            )
            print("Used pandas/pyarrow Parquet fallback:", fallback_file)
            return "pandas/pyarrow"
        raise
    return "Spark"

how = run_step(
    "Parquet write",
    "write.parquet()",
    write_parquet,
    df=monthly_summary,
    note="Spark write is an action. Output files depend on output partitions and the write plan."
)
print("Written by:", how)
print("Output directory:", out_path)

parquet_files = sorted(p for p in parquet_dir.rglob("*.parquet") if p.is_file())
print("Parquet files created:", len(parquet_files))
for file in parquet_files:
    print(" -", file.relative_to(OUT))
if not parquet_files:
    raise RuntimeError(
        "No .parquet data files were found. Check the Spark error above, "
        "write permissions, and the output path."
    )

# Read-back verification: a successful write must be readable by Spark.
try:
    saved_summary = spark.read.parquet(out_path)
except Exception as exc:
    message = str(exc)

    if platform.system() == "Windows" and (
        "UnsatisfiedLinkError" in message
        or "NativeIO$Windows.access0" in message
    ):
        try:
            import pyarrow  # noqa: F401
        except ImportError as pandas_error:
            raise RuntimeError(
            "Install pandas and pyarrow to read Parquet without Hadoop native libraries:\n"
            "    pip install pandas pyarrow"
            ) from pandas_error

        pdf = pd.read_parquet(out_path, engine="pyarrow")
        saved_summary = spark.createDataFrame(
            pdf,
            schema=monthly_summary.schema
        )
        print("Used pandas/pyarrow Parquet read fallback.")
    else:
        raise
print("Rows read back:", saved_summary.count())
saved_summary.printSchema()
saved_summary.show(5, truncate=False)
assert saved_summary.schema == monthly_summary.schema, (
    "Parquet schema does not match the source DataFrame schema."
)
print("Parquet write/read-back verification passed.")


c:\Users\PAVANI\OneDrive\Desktop\OneDrive\Desktop\olist-pandas-assignment\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Used pandas/pyarrow Parquet fallback: C:\Users\PAVANI\OneDrive\Desktop\OneDrive\Desktop\olist-pandas-assignment\output\monthly_sales_summary\part-00000.parquet
Written by: pandas/pyarrow
Output directory: C:\Users\PAVANI\OneDrive\Desktop\OneDrive\Desktop\olist-pandas-assignment\output\monthly_sales_summary
Parquet files created: 1
 - monthly_sales_summary\part-00000.parquet


c:\Users\PAVANI\OneDrive\Desktop\OneDrive\Desktop\olist-pandas-assignment\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\PAVANI\OneDrive\Desktop\OneDrive\Desktop\olist-pandas-assignment\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Used pandas/pyarrow Parquet read fallback.
Rows read back: 23
root
 |-- purchase_year: integer (nullable = true)
 |-- purchase_month: integer (nullable = true)
 |-- distinct_orders: long (nullable = false)
 |-- total_item_sales: double (nullable = true)

+-------------+--------------+---------------+----------------+
|purchase_year|purchase_month|distinct_orders|total_item_sales|
+-------------+--------------+---------------+----------------+
|2016         |9             |1              |134.97          |
|2016         |10            |265            |40325.11        |
|2016         |12            |1              |10.9            |
|2017         |1             |750            |111798.36       |
|2017         |2             |1653           |234223.4        |
+-------------+--------------+---------------+----------------+
only showing top 5 rows
Parquet write/read-back verification passed.


- **Parquet** = column-based compressed file format.
- Good for analytics: reads only needed columns, small size, keeps schema.
- Usually one file per partition (each write task writes its own file), but empty partitions or settings can change this.
- Too many tiny files = slow reads and heavy metadata. Fix: `coalesce(n)` or `repartition(n)` before writing.

## Part 4 - Business outputs

### 4.1 Finance - item sales by category and month

In [24]:
# Which categories have no translation?
(products.join(translation, "product_category_name", "left")
    .filter(col("product_category_name").isNotNull() & col("product_category_name_english").isNull())
    .select("product_category_name").distinct().show(truncate=False))

+---------------------------------------------+
|product_category_name                        |
+---------------------------------------------+
|pc_gamer                                     |
|portateis_cozinha_e_preparadores_de_alimentos|
+---------------------------------------------+



**Join choice: LEFT join** for translation. An inner join would drop sales of untranslated categories. With left join we keep them and use the Portuguese name (or `unknown` if the category is null).
`price` only, freight is excluded.

In [25]:
finance_df = (
    finance_base
    .join(products.select("product_id", "product_category_name"), "product_id", "left")
    .join(translation, "product_category_name", "left")
    .withColumn("purchase_ts", to_timestamp(col("order_purchase_timestamp")))
    .withColumn("reporting_month", date_format(col("purchase_ts"), "yyyy-MM"))
    .withColumn("product_category",
                coalesce(col("product_category_name_english"), col("product_category_name"), lit("unknown")))
    .groupBy("reporting_month", "product_category")
    .agg(spark_round(spark_sum("price"), 2).alias("total_item_sales"),
         countDistinct("order_id").alias("distinct_orders"))
    .orderBy("reporting_month", "product_category")
)
finance_df.explain(True)   # plan only, does not run the query

== Parsed Logical Plan ==
'Sort ['reporting_month ASC NULLS FIRST, 'product_category ASC NULLS FIRST], true
+- Aggregate [reporting_month#1037, product_category#1038], [reporting_month#1037, product_category#1038, round(sum(price#99), 2) AS total_item_sales#1039, count(distinct order_id#81) AS distinct_orders#1040L]
   +- Project [product_category_name#102, product_id#96, order_id#81, customer_id#82, order_status#83, order_purchase_timestamp#84, order_approved_at#85, order_delivered_carrier_date#86, order_delivered_customer_date#87, order_estimated_delivery_date#88, order_item_id#95, seller_id#97, shipping_limit_date#98, price#99, freight_value#100, product_category_name_english#111, purchase_ts#1036, reporting_month#1037, coalesce(product_category_name_english#111, product_category_name#102, unknown) AS product_category#1038]
      +- Project [product_category_name#102, product_id#96, order_id#81, customer_id#82, order_status#83, order_purchase_timestamp#84, order_approved_at#85, orde

In [26]:
# Finance output
# finance_base was already cached and materialized in Part 3 (cell 3.2).
# Do not repeatedly persist/unpersist or clear the Spark cache here.
from pyspark import StorageLevel

if not finance_base.is_cached:
    finance_base.persist(StorageLevel.MEMORY_AND_DISK)
    finance_base.count()  # materialize once if the cache was previously removed

run_step(
    "Finance output",
    "show(20)",
    lambda: finance_df.show(20, truncate=False),
    df=finance_df,
    cache=str(finance_base.storageLevel),
    note="finance_base is cached once; finance_df is evaluated from that stable lineage."
)

+---------------+------------------------+----------------+---------------+
|reporting_month|product_category        |total_item_sales|distinct_orders|
+---------------+------------------------+----------------+---------------+
|2016-09        |health_beauty           |134.97          |1              |
|2016-10        |air_conditioning        |1349.29         |4              |
|2016-10        |audio                   |156.99          |2              |
|2016-10        |auto                    |1128.26         |8              |
|2016-10        |baby                    |1371.17         |9              |
|2016-10        |bed_bath_table          |478.99          |5              |
|2016-10        |computers_accessories   |669.02          |12             |
|2016-10        |consoles_games          |3619.36         |7              |
|2016-10        |cool_stuff              |1046.1          |7              |
|2016-10        |diapers_and_hygiene     |134.9           |1              |
|2016-10    

Open the **SQL tab** now and take the query plan screenshot.

In [27]:
# Validation
fin_total = (
    finance_base
    .agg(spark_round(spark_sum("price"), 2).alias("total_item_sales"))
    .first()["total_item_sales"] # pyright: ignore[reportOptionalSubscript]
)
fin_combos = finance_df.count()
check_total = (orders.filter(col("order_status") == "delivered")
               .join(order_items, "order_id")
               .agg(spark_round(spark_sum("price"), 2)).first()[0]) # pyright: ignore[reportOptionalSubscript]
print("Finance total item sales :", fin_total)
print("Independent check total  :", check_total, "| match:", fin_total == check_total)
print("Month/category combos    :", fin_combos)

# Type your Assignment 3 numbers here to compare (leave None to skip)
A3_TOTAL_SALES = None
A3_COMBOS = None
if A3_TOTAL_SALES is not None:
    print("Matches Assignment 3 total:", round(A3_TOTAL_SALES, 2) == fin_total)
if A3_COMBOS is not None:
    print("Matches Assignment 3 combos:", A3_COMBOS == fin_combos)

Finance total item sales : 13221498.11
Independent check total  : 13221498.11 | match: True
Month/category combos    : 1273


In [28]:
# Release the finance cache only after finance output and validation are complete.
finance_base.unpersist(blocking=True)
spark.catalog.clearCache()
print("is_cached after cleanup:", finance_base.is_cached)

is_cached after cleanup: False


### 4.2 Operations - delivery performance by customer state

**Rules used:**
- delivered date is null -> `not_delivered` (never on_time)
- estimated date is null (but delivered) -> `no_estimate`: counted as delivered, not late
- delivered after estimated -> `late`, else `on_time`
- `late_rate = late_orders / delivered_orders`. Undelivered are excluded from both.
- Group with zero delivered orders -> `late_rate` is null (no divide by zero).

In [29]:
ops_base = (
    orders.join(customers.select("customer_id", "customer_state"), "customer_id", "left")
    .withColumn("purchase_ts", to_timestamp(col("order_purchase_timestamp")))
    .withColumn("delivered_ts", to_timestamp(col("order_delivered_customer_date")))
    .withColumn("estimated_ts", to_timestamp(col("order_estimated_delivery_date")))
    .withColumn("reporting_month", date_format(col("purchase_ts"), "yyyy-MM"))
    .withColumn("delivery_status",
        when(col("delivered_ts").isNull(), "not_delivered")
        .when(col("estimated_ts").isNull(), "no_estimate")
        .when(col("delivered_ts") > col("estimated_ts"), "late")
        .otherwise("on_time"))
)
ops_base.groupBy("delivery_status").count().show()

+---------------+-----+
|delivery_status|count|
+---------------+-----+
|  not_delivered| 2965|
|        on_time|88649|
|           late| 7827|
+---------------+-----+



In [30]:
# Operations output: count delivered orders only when the order was delivered.
# no_estimate still counts as delivered; not_delivered is excluded.
operations_df = (
    ops_base
    .groupBy("reporting_month", "customer_state")
    .agg(
        spark_sum(
            when(col("delivery_status").isin("late", "on_time", "no_estimate"), 1)
            .otherwise(0)
        ).alias("delivered_orders"),
        spark_sum(
            when(col("delivery_status") == "late", 1).otherwise(0)
        ).alias("late_orders")
    )
    .withColumn(
        "late_rate",
        when(
            col("delivered_orders") > 0,
            spark_round(col("late_orders") / col("delivered_orders"), 4)
        ).otherwise(lit(None).cast("double"))
    )
    .orderBy("reporting_month", "customer_state")
)

run_step(
    "Operations output",
    "show()",
    lambda: operations_df.show(20, truncate=False),
    df=operations_df,
    note="Grouped by month and state; undelivered orders are excluded from delivered and late counts."
)

+---------------+--------------+----------------+-----------+---------+
|reporting_month|customer_state|delivered_orders|late_orders|late_rate|
+---------------+--------------+----------------+-----------+---------+
|2016-09        |RR            |0               |0          |NULL     |
|2016-09        |RS            |0               |0          |NULL     |
|2016-09        |SP            |1               |1          |1.0      |
|2016-10        |AL            |1               |0          |0.0      |
|2016-10        |BA            |3               |0          |0.0      |
|2016-10        |CE            |6               |0          |0.0      |
|2016-10        |DF            |6               |0          |0.0      |
|2016-10        |ES            |3               |0          |0.0      |
|2016-10        |GO            |7               |0          |0.0      |
|2016-10        |MA            |4               |1          |0.25     |
|2016-10        |MG            |35              |0          |0.0

In [31]:
from pyspark.sql.functions import sum as spark_sum, col

# Check that operations_df exists before calculating totals
if "operations_df" not in globals():
    raise NameError(
        "operations_df is not defined. "
        "First run the cell that creates operations_df."
    )

# Calculate totals using named columns
tot = (
    operations_df
    .agg(
        spark_sum("delivered_orders").alias("total_delivered_orders"),
        spark_sum("late_orders").alias("total_late_orders")
    )
    .first()
)

if tot is not None:
    total_delivered = tot["total_delivered_orders"] or 0
    total_late = tot["total_late_orders"] or 0

    print("Total delivered orders:", total_delivered)
    print("Total late orders     :", total_late)

# Display results for São Paulo state
operations_df.filter(
    col("customer_state") == "SP"
).show(6, truncate=False)

# Enter your Assignment 3 expected late total here.
# Keep None to skip the comparison.
A3_LATE_TOTAL = None

if A3_LATE_TOTAL is not None and tot is not None:
    actual_late_total = tot["total_late_orders"] or 0

    print(
        "Matches Assignment 3 late total:",
        A3_LATE_TOTAL == actual_late_total
    )

    assert A3_LATE_TOTAL == actual_late_total, (
        f"Expected {A3_LATE_TOTAL}, got {actual_late_total}"
    )

Total delivered orders: 96476
Total late orders     : 7827
+---------------+--------------+----------------+-----------+---------+
|reporting_month|customer_state|delivered_orders|late_orders|late_rate|
+---------------+--------------+----------------+-----------+---------+
|2016-09        |SP            |1               |1          |1.0      |
|2016-10        |SP            |95              |2          |0.0211   |
|2017-01        |SP            |283             |8          |0.0283   |
|2017-02        |SP            |601             |19         |0.0316   |
|2017-03        |SP            |966             |31         |0.0321   |
|2017-04        |SP            |873             |73         |0.0836   |
+---------------+--------------+----------------+-----------+---------+
only showing top 6 rows


- Undelivered is not on-time: the order has not arrived yet, so we do not know the result.
- Denominator is delivered orders because only those can be judged late or on time.
- If either date is null the order is not counted as late.

## Spark UI observations

In [32]:
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
pd.DataFrame(log)

,Step,Action,Jobs,Stages,Tasks,Shuffle/Exchange?,Cache?,Seconds,Observation
0,Infer schema (orders),read.csv(inferSchema=True),4,4,12,Yes,No,0.58,Reading with inferSchema already created a job (it scanned the file).
1,Count orders,count(),2,3,11,Yes,No,0.83,Single scan of the CSV. Task count follows input file splits.
2,Select count,count(),2,3,11,Yes,No,0.22,Select is lazy. Job only started at count.
3,Delivered filter,count(),2,3,11,Yes,No,0.64,Filter is narrow. Each task filters its own partition.
4,Join orders/items,count(),3,4,14,Yes,No,0.98,"Join needs data with the same order_id together, so Exchange appears unless ..."
5,Monthly aggregation,show(),4,7,26,Yes,No,2.02,groupBy and orderBy cause Exchange. Several stages.
6,Deduplication,count(),5,9,26,Yes,No,0.85,dropDuplicates is wide (shuffle).
7,Action count(),count(),4,7,20,Yes,No,0.62,
8,Action show(5),show(5),4,7,26,Yes,No,0.78,
9,Action first(),first(),4,7,26,Yes,No,0.68,


In [33]:
# Input / shuffle / task-duration numbers straight from the Spark UI API
import json, urllib.request

def ui_get(path):
    with urllib.request.urlopen(sc.uiWebUrl + "/api/v1" + path, timeout=10) as r: # pyright: ignore[reportOptionalOperand]
        return json.loads(r.read())

try:
    app_id = sc.applicationId
    stages = ui_get("/applications/%s/stages" % app_id)
    rows = []
    for s in stages:
        if s["status"] != "COMPLETE":
            continue
        row = {"stage": s["stageId"], "name": s["name"][:40], "tasks": s["numTasks"],
               "input_MB": round(s["inputBytes"] / 1e6, 2), "input_records": s["inputRecords"],
               "shuffle_read_MB": round(s["shuffleReadBytes"] / 1e6, 2),
               "shuffle_write_MB": round(s["shuffleWriteBytes"] / 1e6, 2)}
        try:
            q = ui_get("/applications/%s/stages/%s/%s/taskSummary?quantiles=0.0,0.5,1.0"
                       % (app_id, s["stageId"], s["attemptId"]))
            t = q["executorRunTime"]
            row["task_ms_min/med/max"] = "%d / %d / %d" % (t[0], t[1], t[2])
        except Exception:
            row["task_ms_min/med/max"] = "n/a"
        rows.append(row)
    ui_df = pd.DataFrame(rows).sort_values("shuffle_read_MB", ascending=False).head(6)
    display(ui_df)
except Exception as e:
    print("Could not read Spark UI API:", e)

,stage,name,tasks,input_MB,input_records,shuffle_read_MB,shuffle_write_MB,task_ms_min/med/max
53,90,count at NativeMethodAccessorImpl.java:0,8,0.0,0,23.32,0.00,1061 / 1062 / 1075
43,109,$anonfun$withThreadLocalCaptured$1 at Co,8,0.0,0,20.40,19.69,271 / 303 / 324
46,104,$anonfun$withThreadLocalCaptured$1 at Co,8,0.0,0,20.40,19.69,551 / 610 / 619
91,20,$anonfun$withThreadLocalCaptured$1 at Co,4,0.0,0,4.15,0.00,269 / 277 / 280
66,63,take at C:\Users\PAVANI\AppData\Local\Te,4,0.0,0,4.15,0.00,90 / 100 / 105
74,49,$anonfun$withThreadLocalCaptured$1 at Co,4,0.0,0,4.15,0.00,111 / 116 / 119


**What the numbers show (3 executions):**
- Join / aggregation stages have shuffle write and shuffle read. Scan-only stages have input but no shuffle.
- If max task time is much bigger than median, work is uneven (skew or one big partition). If similar, work is balanced.
- On this small local data, times are short, so overhead matters more than real work.

## Spark UI screenshots (required before submission)

Capture these screenshots from **your own running Spark UI** and save them in the repository-root `screenshots/` folder with the exact filenames below. Do not use fabricated or copied screenshots. The images will render when the files exist.

1. `screenshots/jobs_stages.png` — Jobs/Stages page showing a multi-stage execution.
2. `screenshots/sql_plan.png` — SQL tab showing the Finance query plan / execution graph.
3. `screenshots/storage_cache.png` — Storage tab while `finance_base` is cached (capture this before running the cleanup cell that calls `unpersist()`).

![Jobs page, multi-stage execution](../screenshots/jobs_stages.png)

*Jobs / Stages page: the aggregation job is split into stages around shuffle boundaries.*

![SQL tab query plan](../screenshots/sql_plan.png)

*SQL tab: the Finance query plan with Exchange (shuffle) and join nodes.*

![Storage tab cached finance_base](../screenshots/storage_cache.png)

*Storage tab: `finance_base` cached in memory or memory-and-disk.*

**Submission check:** if any image above is missing or broken, the screenshot requirement is not complete yet.

## Engineering reflection

1. **Lazy evaluation:** `select`, `filter` and three `withColumn` calls became one Project/Filter step in the plan. Benefit: one pass over the data, and Spark can skip unneeded columns.
2. **Narrow vs wide:** `filter` works on each partition alone, no data moves. `groupBy` needs same keys together, so it shuffles. Shuffle costs disk, network and time.
3. **DAG and stages:** A shuffle ends one stage and starts the next. The next stage cannot start until the shuffle data is ready.
4. **Partitions and tasks:** Task count = partitions of that stage. It changes with file size, settings (e.g. `shuffle.partitions`), `repartition` and AQE. So it is not fixed for a DataFrame.
5. **Caching:** I cached `finance_base` (delivered orders joined to items), used it in Finance, and the Storage tab listed it with its size in memory. Caching is wasteful if the data is used only once or is too big for memory.
6. **Join strategy:** Check the `explain()` output above. Usually Sort Merge Join for big tables and Broadcast Hash Join for small ones. It depends on table size, the broadcast threshold, join type and hints.
7. **Spark vs PostgreSQL:** For this small data, PostgreSQL is simpler and faster (no startup, indexes, one machine is enough). Spark is worth it when data is too big for one machine, needs many files (Parquet), heavy transforms, or must scale out.
8. **Spark UI:** First look at Stages (which one is slow), then task time min/median/max (skew), shuffle read/write size, spill to disk, and the SQL tab plan (bad joins, extra Exchange).
9. **Data quality:** Duplicates change sums and counts. Without a known grain and a clear rule, `dropDuplicates` may keep the wrong row and the numbers silently go wrong.
10. **Business definitions:** An order with no delivered date is labelled `not_delivered`, never `on_time`. It is excluded from both numerator and denominator of `late_rate`.

## Improved answers to the initial questions
1. The source is read once for two filters and a count, because the filters are merged. It can be read again if I run another action without cache.
2. A DAG shows the order and dependencies of all steps. Spark uses it to optimise, split into stages and rerun lost parts.
3. A stage is a set of steps between shuffles. A task is that stage's work on one partition. One stage has many tasks.
4. `filter` needs only the row itself. `groupBy` needs all rows of the same key together, which are spread over many partitions, so a shuffle is needed.

## Optional final step — save Finance and Operations outputs

The assignment's required Parquet exercise is the monthly summary in Part 3. This extra cell also saves the two business outputs for convenient reuse. It overwrites only these generated output folders.

In [ ]:
finance_out_path = str((OUT / "finance_by_category_month").resolve())
operations_out_path = str((OUT / "operations_by_state_month").resolve())

run_step(
    "Write Finance Parquet",
    "write.parquet()",
    lambda: finance_df.write.mode("overwrite").parquet(finance_out_path),
    df=finance_df,
    note="Finance output saved as Parquet."
)
run_step(
    "Write Operations Parquet",
    "write.parquet()",
    lambda: operations_df.write.mode("overwrite").parquet(operations_out_path),
    df=operations_df,
    note="Operations output saved as Parquet."
)

finance_saved = spark.read.parquet(finance_out_path)
operations_saved = spark.read.parquet(operations_out_path)
print("Finance Parquet rows:", finance_saved.count())
print("Operations Parquet rows:", operations_saved.count())
print("Finance path:", finance_out_path)
print("Operations path:", operations_out_path)
